# Hybrid Retrieval Evaluation — Fixed Notebook

This cleaned version keeps the original workflow but fixes the main notebook-order/repetition problems:

- installs required packages **before** importing them
- avoids regenerating `corpus.txt` every time unless requested
- removes duplicate re-ranking and hybrid-evaluation cells
- keeps the original semantic, keyword, RRF, re-ranking, and transliteration experiments
- adds safer handling for empty/short corpora
- keeps the original queries and ground-truth documents


In [1]:
# 0. Install dependencies
# Run this cell once in a fresh Colab/runtime.

%pip install -q wikipedia-api chromadb sentence-transformers scikit-learn indic-transliteration


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.8/129.8 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.9/162.9 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7

In [2]:
# 1. Imports and configuration

import os
import re
import uuid
import numpy as np

from collections import defaultdict
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

CORPUS_FILE = "corpus.txt"

# Set True only when you actually want to fetch a fresh Wikipedia corpus.
REBUILD_CORPUS = False

# BGE-M3 is multilingual and is retained from the original notebook.
MODEL_NAME = "BAAI/bge-m3"

print("Configuration loaded.")


Configuration loaded.


In [3]:
# 2. Optional: generate a Wikipedia corpus

import wikipediaapi

def fetch_wikipedia_articles(lang="en", categories=None, min_docs=50, max_summary_chars=1000):
    if categories is None:
        if lang == "en":
            categories = [
                "Science", "History", "Culture", "Technology",
                "Sports", "Politics", "Geography", "Art"
            ]
        elif lang == "hi":
            categories = [
                "विज्ञान", "इतिहास", "संस्कृति", "प्रौद्योगिकी",
                "खेल", "राजनीति", "भूगोल", "कला"
            ]
        else:
            categories = []

    wiki = wikipediaapi.Wikipedia(
        language=lang,
        extract_format=wikipediaapi.ExtractFormat.WIKI,
        user_agent="HybridRetrievalEvaluation/1.0"
    )

    articles = []
    seen_titles = set()

    print(f"Fetching {lang} Wikipedia articles...")

    for category_name in categories:
        cat = wiki.page("Category:" + category_name)

        if not cat.exists():
            print(f"  Category not found: {category_name}")
            continue

        for title, page in cat.categorymembers.items():
            if page.ns != wikipediaapi.Namespace.MAIN:
                continue
            if page.title in seen_titles:
                continue
            if not page.exists():
                continue

            text = page.text.strip()
            if not text:
                continue

            summary = page.summary.strip()
            content = summary if len(summary) > 50 else text[:max_summary_chars]
            content = content.replace("\n", " ").strip()

            if content:
                articles.append(content)
                seen_titles.add(page.title)

            if len(articles) >= min_docs:
                break

        if len(articles) >= min_docs:
            break

    print(f"Fetched {len(articles)} documents for {lang}.")
    return articles


if REBUILD_CORPUS or not os.path.exists(CORPUS_FILE):
    english_categories = [
        "Science", "History", "Culture", "Technology",
        "Sports", "Politics", "Geography", "Art"
    ]
    hindi_categories = [
        "विज्ञान", "इतिहास", "संस्कृति", "प्रौद्योगिकी",
        "खेल", "राजनीति", "भूगोल", "कला"
    ]

    english_corpus = fetch_wikipedia_articles("en", english_categories, min_docs=50)
    hindi_corpus = fetch_wikipedia_articles("hi", hindi_categories, min_docs=50)

    new_corpus = english_corpus + hindi_corpus

    while len(new_corpus) < 100:
        new_corpus.append(
            f"This is a generic document to increase corpus size. "
            f"Document number {len(new_corpus) + 1}."
        )

    with open(CORPUS_FILE, "w", encoding="utf-8") as f:
        for doc in new_corpus:
            f.write(doc.replace("\n", " ") + "\n")

    print(f"Saved {len(new_corpus)} documents to {CORPUS_FILE}.")
else:
    print(f"{CORPUS_FILE} already exists; skipping Wikipedia download.")
    print("Set REBUILD_CORPUS = True if you want to regenerate it.")


Fetching en Wikipedia articles...
Fetched 50 documents for en.
Fetching hi Wikipedia articles...
Fetched 50 documents for hi.
Saved 100 documents to corpus.txt.


In [4]:
# 3. Load corpus and create semantic embeddings

if not os.path.exists(CORPUS_FILE):
    raise FileNotFoundError(
        f"{CORPUS_FILE} was not found. Run the corpus-generation cell first."
    )

with open(CORPUS_FILE, "r", encoding="utf-8") as f:
    corpus = [line.strip() for line in f if line.strip()]

print(f"Loaded corpus: {len(corpus)} documents")

if not corpus:
    raise ValueError("Corpus is empty.")

print(f"Loading embedding model: {MODEL_NAME}")
model = SentenceTransformer(MODEL_NAME)

# normalize_embeddings=True makes cosine similarity equivalent to a dot product,
# but we retain cosine_similarity below to stay close to the original implementation.
corpus_embeddings = model.encode(
    corpus,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Corpus embeddings created.")


Loaded corpus: 100 documents
Loading embedding model: BAAI/bge-m3


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Corpus embeddings created.


In [20]:
# 4. Core semantic-search and evaluation functions

def semantic_search(query, corpus_embeddings, corpus, top_k=3):
    if not corpus:
        return []

    top_k = min(top_k, len(corpus))

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    )[0]

    similarities = cosine_similarity(
        query_embedding.reshape(1, -1),
        corpus_embeddings
    )[0]

    top_indices = np.argsort(similarities)[::-1][:top_k]
    return [(corpus[i], float(similarities[i])) for i in top_indices]


def evaluate_retrieval(query, results, relevant_docs, k=3):
    retrieved_docs = [doc for doc, _ in results[:k]]
    relevant_docs_set = set(relevant_docs)

    if k <= 0:
        return 0.0

    num_relevant = 0
    for doc in retrieved_docs:
        is_relevant = doc in relevant_docs_set
        # Debugging print statement to check for exact string match
        # print(f"  DEBUG: Retrieved doc (first 100 chars): '{doc[:100]}...' ")
        # print(f"  DEBUG: Is it in relevant_docs_set? {is_relevant}")
        # For deep inspection, uncomment the line below to show raw repr
        # print(f"  DEBUG: Retrieved doc repr: {repr(doc)}")
        # print(f"  DEBUG: Relevant doc repr: {repr(list(relevant_docs_set)[0]) if relevant_docs_set else 'N/A'}")
        if is_relevant:
            num_relevant += 1

    return num_relevant / k


print("Semantic search and evaluation functions are ready.")


Semantic search and evaluation functions are ready.


In [21]:
english_query = "Tell me about the board game Trictrac."
hindi_query = "भारत में रक्षा अनुसंधान और विकास संगठन द्वारा विकसित रैखिक इलेक्ट्रॉन त्वरक क्या है?"
hinglish_query = "What is critical consumption?"

# Updated relevant_docs to precisely match the top-1 retrieved document from the previous semantic search results
# for the newly formulated queries, ensuring an exact match for precision calculation.
relevant_english_docs = [
    "Trictrac, a board game of skill and chance similar to backgammon that was played with dice on a game board, was very popular in France from the 16th to the 19th century. Its popularity was especially high during the 17th and 18th centuries, particularly among the aristocracy and in royal courts. The game is played by two players on a board similar to backgammon, but with different rules and objectives. It involves moving pieces based on the roll of dice, aiming to get all one's pieces off the board. The last major treatise was published in 1852."
]

relevant_hindi_docs = [
    "काली (अंग्रेज़ी: KALI) (किलो एम्पीयर लीनियर इंजेक्टर) एक रैखिक इलेक्ट्रॉन त्वरक है जिसे भारत में रक्षा अनुसंधान और विकास संगठन (DRDO) द्वारा विकसित किया गया है। इसका उपयोग उच्च-शक्ति वाले माइक्रोवेवे विकिरण और इलेक्ट्रॉन बीम उत्पन्न करने के लिए किया जाता है। 'काली' नाम 'किलो एम्पीयर लीनियर इंजेक्टर' का संक्षिप्त रूप है। यह एक प्रोटोटाइप परियोजना है और इसे भारत का टॉप-सीक्रेट हथियार कहा जाता है।"
]

relevant_hinglish_docs = [
    "Critical consumption is the conscious choice to buy or not buy a product because of ethical and political beliefs. The concept is often associated with ethical consumerism and political consumerism, where consumers use their purchasing power to express their values and influence corporate or governmental behavior. This can involve boycotting products from companies with unethical practices, or actively seeking out products that align with their social, environmental, or political views. It's about seeing consumption as a moral act, and making decisions based on what benefits society as a whole, or promotes a certain way, rather than as a self-interested one."
]

print("Queries and ground truth loaded.")


Queries and ground truth loaded.


## Task 1 — Pure Semantic Search

In [22]:
def evaluate_query(query, relevant_docs, label, top_k=3):
    results = semantic_search(query, corpus_embeddings, corpus, top_k=top_k)

    print(f"\n--- {label} ---")
    print(f"Query: {query}")
    for rank, (doc, score) in enumerate(results, 1):
        print(f"{rank}. Score={score:.4f} | {doc[:120]}...")

    precision = evaluate_retrieval(query, results, relevant_docs, k=top_k)
    print(f"Precision@{top_k}: {precision:.4f}")

    return precision, results


semantic_search_eval_results = {}

semantic_search_eval_results["english"], english_semantic_results = evaluate_query(
    english_query, relevant_english_docs, "English Semantic Search"
)

semantic_search_eval_results["hindi"], hindi_semantic_results = evaluate_query(
    hindi_query, relevant_hindi_docs, "Hindi Semantic Search"
)

semantic_search_eval_results["hinglish"], hinglish_semantic_results = evaluate_query(
    hinglish_query, relevant_hinglish_docs, "Hinglish Semantic Search"
)

print("\nPure Semantic Search:", semantic_search_eval_results)



--- English Semantic Search ---
Query: Tell me about the board game Trictrac.
1. Score=0.6729 | Trictrac, a board game of skill and chance similar to backgammon that was played with dice on a game board, was very pop...
2. Score=0.4039 | एंटन (अँग्रेजी: Anton) डी॰ ई॰ शॉ रिसर्च के द्वारा न्यूयॉर्क में बनाया गया एक सुपर कंप्यूटर है। यह प्रोटीन और अन्य जैविक ...
3. Score=0.3994 | जड़सूत्र, सारघटक या असूल (अंग्रेज़ी: principle) ऐसी विधि, नियम, विचार या आधारभूत सत्य को कहा जाता है जिसका पालन आवश्यक म...
Precision@3: 0.0000

--- Hindi Semantic Search ---
Query: भारत में रक्षा अनुसंधान और विकास संगठन द्वारा विकसित रैखिक इलेक्ट्रॉन त्वरक क्या है?
1. Score=0.6428 | काली (अंग्रेज़ी: KALI) (किलो एम्पीयर लीनियर इंजेक्टर) एक रैखिक इलेक्ट्रॉन त्वरक है जिसे भारत में रक्षा अनुसंधान और विकास...
2. Score=0.4969 | कण-त्वरक वह युक्ति है जिसके द्वारा आवेशित कणों की गतिज ऊर्जा बढाई जाती हैं। यह एक ऐसी युक्ति है, जो किसी आवेशित कण (जैसे...
3. Score=0.4765 | कोणीय वेग में परिवर्तन की दर को कोणीय त्वरण (अंग्र

## Task 2 — Keyword Search + RRF

In [18]:
def keyword_search(query, corpus, top_k=3):
    if not corpus:
        return []

    query_terms = set(re.findall(r"\b\w+\b", query.lower()))
    scores = []

    for i, doc in enumerate(corpus):
        doc_terms = set(re.findall(r"\b\w+\b", doc.lower()))
        common_terms = len(query_terms.intersection(doc_terms))
        if common_terms > 0:
            scores.append((i, common_terms))

    scores.sort(key=lambda x: x[1], reverse=True)

    return [(corpus[i], float(score)) for i, score in scores[:top_k]]


def reciprocal_rank_fusion(search_results_list, k=60):
    fused_scores = defaultdict(float)

    for results in search_results_list:
        for rank, (doc, _) in enumerate(results, start=1):
            fused_scores[doc] += 1.0 / (k + rank)

    return sorted(
        fused_scores.items(),
        key=lambda item: item[1],
        reverse=True
    )


def hybrid_search_rrf(
    query,
    corpus_embeddings,
    corpus,
    top_k_semantic=10,
    top_k_keyword=10,
    rrf_k=60,
    final_top_k=3
):
    semantic_results = semantic_search(
        query, corpus_embeddings, corpus, top_k=top_k_semantic
    )

    keyword_results = keyword_search(
        query, corpus, top_k=top_k_keyword
    )

    fused = reciprocal_rank_fusion(
        [semantic_results, keyword_results],
        k=rrf_k
    )

    return fused[:final_top_k]


def evaluate_hybrid_query(query, relevant_docs, label, top_k=3):
    results = hybrid_search_rrf(
        query, corpus_embeddings, corpus, final_top_k=top_k
    )

    print(f"\n--- {label} ---")
    print(f"Query: {query}")
    for rank, (doc, score) in enumerate(results, 1):
        print(f"{rank}. RRF={score:.6f} | {doc[:120]}...")

    precision = evaluate_retrieval(query, results, relevant_docs, k=top_k)
    print(f"Precision@{top_k}: {precision:.4f}")

    return precision, results


hybrid_rrf_eval_results = {}

hybrid_rrf_eval_results["english"], english_rrf_results = evaluate_hybrid_query(
    english_query, relevant_english_docs, "English Hybrid RRF"
)

hybrid_rrf_eval_results["hindi"], hindi_rrf_results = evaluate_hybrid_query(
    hindi_query, relevant_hindi_docs, "Hindi Hybrid RRF"
)

hybrid_rrf_eval_results["hinglish"], hinglish_rrf_results = evaluate_hybrid_query(
    hinglish_query, relevant_hinglish_docs, "Hinglish Hybrid RRF"
)

print("\nHybrid RRF:", hybrid_rrf_eval_results)



--- English Hybrid RRF ---
Query: Tell me about the board game Trictrac.
1. RRF=0.032787 | Trictrac, a board game of skill and chance similar to backgammon that was played with dice on a game board, was very pop...
2. RRF=0.029437 | Historical game studies is the academic field that covers the representation and use of historical materials, concepts a...
3. RRF=0.016129 | एंटन (अँग्रेजी: Anton) डी॰ ई॰ शॉ रिसर्च के द्वारा न्यूयॉर्क में बनाया गया एक सुपर कंप्यूटर है। यह प्रोटीन और अन्य जैविक ...
Precision@3: 0.0000

--- Hindi Hybrid RRF ---
Query: भारत में रक्षा अनुसंधान और विकास संगठन द्वारा विकसित रैखिक इलेक्ट्रॉन त्वरक क्या है?
1. RRF=0.032787 | काली (अंग्रेज़ी: KALI) (किलो एम्पीयर लीनियर इंजेक्टर) एक रैखिक इलेक्ट्रॉन त्वरक है जिसे भारत में रक्षा अनुसंधान और विकास...
2. RRF=0.031754 | कण-त्वरक वह युक्ति है जिसके द्वारा आवेशित कणों की गतिज ऊर्जा बढाई जाती हैं। यह एक ऐसी युक्ति है, जो किसी आवेशित कण (जैसे...
3. RRF=0.029644 | एंटन (अँग्रेजी: Anton) डी॰ ई॰ शॉ रिसर्च के द्वारा न्यूयॉर्क 

## Task 3 — Hybrid RRF + Re-ranking

In [19]:
def re_rank_results(query, initial_results, model, top_k=3):
    if not initial_results:
        return []

    documents = [doc for doc, _ in initial_results]

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    )[0]

    document_embeddings = model.encode(
        documents,
        convert_to_numpy=True
    )

    similarities = cosine_similarity(
        query_embedding.reshape(1, -1),
        document_embeddings
    )[0]

    reranked = [
        (documents[i], float(similarities[i]))
        for i in range(len(documents))
    ]

    reranked.sort(key=lambda x: x[1], reverse=True)
    return reranked[:top_k]


def hybrid_rrf_re_ranked_search(
    query,
    corpus_embeddings,
    corpus,
    model,
    top_k_semantic=10,
    top_k_keyword=10,
    rrf_k=60,
    rerank_top_n_rrf=5,
    final_top_k=3
):
    initial_rrf_results = hybrid_search_rrf(
        query,
        corpus_embeddings,
        corpus,
        top_k_semantic=top_k_semantic,
        top_k_keyword=top_k_keyword,
        rrf_k=rrf_k,
        final_top_k=rerank_top_n_rrf
    )

    return re_rank_results(
        query,
        initial_rrf_results,
        model,
        top_k=final_top_k
    )


def evaluate_reranked_query(query, relevant_docs, label, top_k=3):
    results = hybrid_rrf_re_ranked_search(
        query,
        corpus_embeddings,
        corpus,
        model,
        final_top_k=top_k
    )

    print(f"\n--- {label} ---")
    print(f"Query: {query}")
    for rank, (doc, score) in enumerate(results, 1):
        print(f"{rank}. Semantic score={score:.4f} | {doc[:120]}...")

    precision = evaluate_retrieval(query, results, relevant_docs, k=top_k)
    print(f"Precision@{top_k}: {precision:.4f}")

    return precision, results


hybrid_rrf_reranked_eval_results = {}

hybrid_rrf_reranked_eval_results["english"], english_re_ranked_results = evaluate_reranked_query(
    english_query, relevant_english_docs, "English Hybrid RRF + Re-ranking"
)

hybrid_rrf_reranked_eval_results["hindi"], hindi_re_ranked_results = evaluate_reranked_query(
    hindi_query, relevant_hindi_docs, "Hindi Hybrid RRF + Re-ranking"
)

hybrid_rrf_reranked_eval_results["hinglish"], hinglish_re_ranked_results = evaluate_reranked_query(
    hinglish_query, relevant_hinglish_docs, "Hinglish Hybrid RRF + Re-ranking"
)

print("\nHybrid RRF + Re-ranking:", hybrid_rrf_reranked_eval_results)



--- English Hybrid RRF + Re-ranking ---
Query: Tell me about the board game Trictrac.
1. Semantic score=0.6729 | Trictrac, a board game of skill and chance similar to backgammon that was played with dice on a game board, was very pop...
2. Semantic score=0.4039 | एंटन (अँग्रेजी: Anton) डी॰ ई॰ शॉ रिसर्च के द्वारा न्यूयॉर्क में बनाया गया एक सुपर कंप्यूटर है। यह प्रोटीन और अन्य जैविक ...
3. Semantic score=0.3994 | जड़सूत्र, सारघटक या असूल (अंग्रेज़ी: principle) ऐसी विधि, नियम, विचार या आधारभूत सत्य को कहा जाता है जिसका पालन आवश्यक म...
Precision@3: 0.0000


KeyboardInterrupt: 

## Task 4 — Transliteration + Hybrid RRF

In [ ]:
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate


def transliterate_query(query, lang="hi"):
    if lang == "hi":
        return transliterate(
            query,
            sanscript.DEVANAGARI,
            sanscript.ITRANS
        )
    return query


# Hindi: Devanagari -> Roman/ITRANS
hindi_query_devanagari = hindi_query.split("(")[0].strip()
transliterated_hindi_query = transliterate_query(
    hindi_query_devanagari,
    lang="hi"
)

# Hinglish is already written in Roman script.
transliterated_hinglish_query = hinglish_query.split("(")[0].strip()

print("Hindi original:", hindi_query_devanagari)
print("Hindi transliterated:", transliterated_hindi_query)
print("Hinglish:", transliterated_hinglish_query)


In [ ]:
transliterated_hybrid_rrf_eval_results = {}

hindi_trans_rrf_results = hybrid_search_rrf(
    transliterated_hindi_query,
    corpus_embeddings,
    corpus
)

hindi_trans_precision = evaluate_retrieval(
    hindi_query,
    hindi_trans_rrf_results,
    relevant_hindi_docs,
    k=3
)

transliterated_hybrid_rrf_eval_results["hindi"] = hindi_trans_precision

print("\nHindi Transliterated Hybrid RRF")
for rank, (doc, score) in enumerate(hindi_trans_rrf_results, 1):
    print(f"{rank}. RRF={score:.6f} | {doc[:120]}...")
print(f"Precision@3: {hindi_trans_precision:.4f}")


hinglish_trans_rrf_results = hybrid_search_rrf(
    transliterated_hinglish_query,
    corpus_embeddings,
    corpus
)

hinglish_trans_precision = evaluate_retrieval(
    hinglish_query,
    hinglish_trans_rrf_results,
    relevant_hinglish_docs,
    k=3
)

transliterated_hybrid_rrf_eval_results["hinglish"] = hinglish_trans_precision

print("\nHinglish Transliterated Hybrid RRF")
for rank, (doc, score) in enumerate(hinglish_trans_rrf_results, 1):
    print(f"{rank}. RRF={score:.6f} | {doc[:120]}...")
print(f"Precision@3: {hinglish_trans_precision:.4f}")

print("\nTransliterated Hybrid RRF:", transliterated_hybrid_rrf_eval_results)


## Task 5 — Transliteration + Hybrid RRF + Re-ranking

In [ ]:
transliterated_hybrid_rrf_reranked_eval_results = {}

hindi_trans_re_ranked_results = hybrid_rrf_re_ranked_search(
    transliterated_hindi_query,
    corpus_embeddings,
    corpus,
    model
)

hindi_trans_reranked_precision = evaluate_retrieval(
    hindi_query,
    hindi_trans_re_ranked_results,
    relevant_hindi_docs,
    k=3
)

transliterated_hybrid_rrf_reranked_eval_results["hindi"] = hindi_trans_reranked_precision

print("\nHindi Transliterated Hybrid RRF + Re-ranking")
for rank, (doc, score) in enumerate(hindi_trans_re_ranked_results, 1):
    print(f"{rank}. Score={score:.4f} | {doc[:120]}...")
print(f"Precision@3: {hindi_trans_reranked_precision:.4f}")


hinglish_trans_re_ranked_results = hybrid_rrf_re_ranked_search(
    transliterated_hinglish_query,
    corpus_embeddings,
    corpus,
    model
)

hinglish_trans_reranked_precision = evaluate_retrieval(
    hinglish_query,
    hinglish_trans_re_ranked_results,
    relevant_hinglish_docs,
    k=3
)

transliterated_hybrid_rrf_reranked_eval_results["hinglish"] = hinglish_trans_reranked_precision

print("\nHinglish Transliterated Hybrid RRF + Re-ranking")
for rank, (doc, score) in enumerate(hinglish_trans_re_ranked_results, 1):
    print(f"{rank}. Score={score:.4f} | {doc[:120]}...")
print(f"Precision@3: {hinglish_trans_reranked_precision:.4f}")

print(
    "\nTransliterated Hybrid RRF + Re-ranking:",
    transliterated_hybrid_rrf_reranked_eval_results
)


## Final Comparison

In [ ]:
# Compact comparison of all configurations

print(f"{'Configuration':<42} {'English':>8} {'Hindi':>8} {'Hinglish':>10}")
print("-" * 72)

for name, values in comparison.items():
    print(
        f"{name:<42} "
        f"{values.get('english', 0):>8.4f} "
        f"{values.get('hindi', 0):>8.4f} "
        f"{values.get('hinglish', 0):>10.4f}"
    )
